# 07 - Evaluation Metrics

In [4]:
import pandas as pd


compare_table = pd.DataFrame({
    "Model": [
        "Baseline Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],

    "R2 Score": [
        0.8249, 
        0.7551,
        0.8105,
        0.7091
    ],


    "MSE":[
        138607181995,
        193847511816,
        149995766209,
        230257944252
    ],


    "MAPE":[
        0.2153,
        0.1712,
        0.1737,
        0.2783
    ],


    "MdAPE":[
        0.1466,
        0.1107,
        0.1131,
        0.1962
    ],

})

compare_table



,Model,R2 Score,MSE,MAPE,MdAPE
0,Baseline Linear Regression,0.8249,138607181995,0.2153,0.1466
1,Decision Tree,0.7551,193847511816,0.1712,0.1107
2,Random Forest,0.8105,149995766209,0.1737,0.1131
3,XGBoost,0.7091,230257944252,0.2783,0.1962


In [5]:
## With Feature Engineering and District Mapping

import pandas as pd


compare_table = pd.DataFrame({
    "Model (With Feature Engineering)": [
        "Baseline Linear Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],

    "R2 Score": [
        0.8251, 
        0.7528,
        0.8383,
        0.8799,
    ],


    "MSE":[
       138411149388,
       195690336208,
       127998459581,
       95029723645
    ],


    "MAPE":[
        0.2154,
        0.1748,
        0.1574,
        0.1385
    ],


    "MdAPE":[
        0.1465,
        0.1087,
        0.0993,
        0.0983
    ],

})

compare_table


,Model (With Feature Engineering),R2 Score,MSE,MAPE,MdAPE
0,Baseline Linear Regression,0.8251,138411149388,0.2154,0.1465
1,Decision Tree,0.7528,195690336208,0.1748,0.1087
2,Random Forest,0.8383,127998459581,0.1574,0.0993
3,XGBoost,0.8799,95029723645,0.1385,0.0983


### Summary 

- Highest R2: Feature Engineered XGBoost
- Lowest MSE: Feature Engineered XGBoost
- Lowest MAPE: Feature Engineered XGBoost
- Lowest MdAPE: Feature Engineered XGBoost


- Best Model: Feature Engineered XG Boost (with slight hyperparamter tuning)

- Notes: Feature engineering and removing outliers helped to increase accuracy for all models

### Saving XGBoost Model

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
import joblib

In [7]:
import geopandas as gpd
from utils import create_time_split, get_preprocessing_pipeline


school_districts = gpd.read_file("california_school_districts.geojson")  
unified_districts = school_districts[school_districts["DistrictType"] == "Unified"].copy()

df = pd.read_csv('Data/cleaned_sold.csv')

properties_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
    crs="EPSG:4326"
)

if properties_gdf.crs != unified_districts.crs:
    properties_gdf = properties_gdf.to_crs(unified_districts.crs)

properties_enriched = gpd.sjoin(
    properties_gdf, 
    unified_districts[['DistrictName', 'geometry']], 
    how="left", 
    predicate="within"  
)

df = properties_enriched.drop(columns=['geometry', 'index_right'])


df['CloseDate'] = pd.to_datetime(df['CloseDate'])

df['BedBathRatio'] = df['BedroomsTotal'] / np.maximum(df['BathroomsTotalInteger'], 1)
df['AgeProperty'] =  (df['CloseDate'].dt.year - df['YearBuilt']).clip(lower=0)

df['BedBathRatio'] = df['BedBathRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)
df['AgeProperty'] = df['AgeProperty'].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df.drop(columns = ["Flooring"]) #too many nulls and weird formatting, had to remove for better performance/less errors

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

def extract_date_features(dataframe, date_col):
    df_feat = dataframe.copy()
    df_feat[f'{date_col}_year'] = df_feat[date_col].dt.year
    df_feat[f'{date_col}_month'] = df_feat[date_col].dt.month
    df_feat[f'{date_col}_day'] = df_feat[date_col].dt.day
    df_feat[f'{date_col}_dayofweek'] = df_feat[date_col].dt.dayofweek
    df_feat = df_feat.drop(columns=[date_col])
    return df_feat

X_train_raw = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test_raw = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

X_train_numeric = extract_date_features(X_train_raw, 'CloseDate')
X_test_numeric = extract_date_features(X_test_raw, 'CloseDate')

preprocessor = get_preprocessing_pipeline(X_train_numeric)
X_train_processed = preprocessor.fit_transform(X_train_numeric)
X_test_processed = preprocessor.transform(X_test_numeric)

Training Window (X=12 months): 2025-05-30 to 2026-05-30 | Rows: 122208
Testing Window (1 month): 2026-05-30 to 2026-06-30


In [9]:
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    eval_metric='rmse'
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

joblib.dump(model, "house_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

MSE: 95029723645.80826
R² Score: 0.8799638496949772
MAPE: 0.13859749555360645
MdAPE: 0.0983686403508772


['preprocessor.pkl']

In [10]:
df.columns

Index(['ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'SubdivisionName', 'YearBuilt',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'StateOrProvince',
       'FireplaceYN', 'Stories', 'Levels', 'LotSizeArea', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'DistrictName', 'BedBathRatio', 'AgeProperty'],
      dtype='object')